# An AI Agent That Uses 2 Tools - Calculator and Weather API 
Create a file langchain_agent_calculator_weather.py

# STEPS

## Work with python3.10. Do not work with higher version.
### Step 1) Create virtual environment (Took 2 minutes):
> C:\Users\hi\AppData\Local\Programs\Python\Python310\python.exe -m venv .venv


### Step 2) Activate virtual envir
> .venv\Scripts\Activate.ps1

### Step 3) Install jupyter (Took 3 minutes):
> python -m pip install jupyter


### Step 4) Run jupyter:
> python -m jupyter notebook


In [9]:
# Check version
import sys, requests

print("Python:", sys.version)
print("Requests:", requests.__version__)

Python: 3.10.11 (tags/v3.10.11:7d4cc5a, Apr  5 2023, 00:38:17) [MSC v.1929 64 bit (AMD64)]
Requests: 2.33.0


### Step 5: Install Following

In [1]:
# Following is compatible with above
!pip install langchain==0.2.14 langchain-community==0.2.12 langchain-openai==0.1.25

  Using cached langchain-0.2.14-py3-none-any.whl (997 kB)
  Using cached langchain_community-0.2.12-py3-none-any.whl (2.3 MB)
  Using cached langchain_openai-0.1.25-py3-none-any.whl (51 kB)
  Using cached langchain_core-0.2.43-py3-none-any.whl (397 kB)
  Using cached numpy-1.26.4-cp310-cp310-win_amd64.whl (15.8 MB)
  Using cached pydantic-2.12.5-py3-none-any.whl (463 kB)
  Using cached langchain_text_splitters-0.2.4-py3-none-any.whl (25 kB)
  Using cached sqlalchemy-2.0.48-cp310-cp310-win_amd64.whl (2.1 MB)
  Using cached tenacity-8.5.0-py3-none-any.whl (28 kB)
  Using cached async_timeout-4.0.3-py3-none-any.whl (5.7 kB)
  Using cached langsmith-0.1.147-py3-none-any.whl (311 kB)
  Using cached aiohttp-3.13.3-cp310-cp310-win_amd64.whl (456 kB)
  Using cached dataclasses_json-0.6.7-py3-none-any.whl (28 kB)
  Using cached tiktoken-0.12.0-cp310-cp310-win_amd64.whl (879 kB)
  Using cached openai-1.109.1-py3-none-any.whl (948 kB)
  Using cached aiosignal-1.4.0-py3-none-any.whl (7.5 kB)
  Usi


[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [10]:

import importlib.metadata

print("langchain:", importlib.metadata.version("langchain"))
print("langchain-community:", importlib.metadata.version("langchain-community"))
print("langchain-openai:", importlib.metadata.version("langchain-openai"))

langchain: 0.2.14
langchain-community: 0.2.12
langchain-openai: 0.1.25


In [11]:
# Check version
import sys, openai

print("OpenAI:", openai.__version__)

OpenAI: 1.109.1


# Step6: Run the code

### Try
```
input1: What is 1 + 2 * 4
input2: What is the current weather condition in Moscow ?
input3: I am trying to calculate this math expression. What is one plus two times four ?
input4: I am living in Moscow. I was thinking of going to the park. I wonder how is weather outside. Is it going going to be nice today ?

In [ ]:
from langchain_openai import ChatOpenAI
from langchain.agents import initialize_agent, AgentType
from langchain.tools import Tool
from langchain.memory import ConversationBufferMemory

import requests
import math

import warnings
from langchain_core._api.deprecation import LangChainDeprecationWarning
warnings.filterwarnings("ignore", category=LangChainDeprecationWarning)

# os.environ["OPENAI_API_KEY"] = "your-key"
OPENAI_API_KEY="key_here"

# -------------------------
# TOOL 1: Calculator
# -------------------------
def calculator(expression: str) -> str:
    print("...Running calculator tool")
    try:
        return str(eval(expression, {"__builtins__": None}, vars(math)))
    except:
        return "Error in calculation"


# -------------------------
# TOOL 2: Weather
# -------------------------
def get_weather(city: str) -> str:
    print("...Running weather tool")
    try:
        geo_url = f"https://geocoding-api.open-meteo.com/v1/search?name={city}"
        geo_res = requests.get(geo_url).json()

        if "results" not in geo_res:
            return "City not found"

        lat = geo_res["results"][0]["latitude"]
        lon = geo_res["results"][0]["longitude"]

        weather_url = f"https://api.open-meteo.com/v1/forecast?latitude={lat}&longitude={lon}&current_weather=true"
        weather_res = requests.get(weather_url).json()

        temp = weather_res["current_weather"]["temperature"]
        wind = weather_res["current_weather"]["windspeed"]

        return f"Temperature: {temp}°C, Wind Speed: {wind} km/h"

    except:
        return "Error fetching weather"


# -------------------------
# MAIN FUNCTION
# -------------------------
def main():

    llm = ChatOpenAI(
        model="gpt-4o-mini",
        temperature=0,
        api_key=OPENAI_API_KEY
    )

    tools = [
        Tool(
            name="Calculator",
            func=calculator,
            description="Useful for solving math expressions like 25*4 or 100/5"
        ),
        Tool(
            name="Weather",
            func=get_weather,
            description="Useful for getting weather of a city like Delhi, London, Moscow"
        )
    ]

    memory = ConversationBufferMemory(
        memory_key="chat_history",
        return_messages=True
    )

    agent = initialize_agent(
        tools=tools,
        llm=llm,
        agent=AgentType.CHAT_ZERO_SHOT_REACT_DESCRIPTION,
        memory=memory,
        verbose=True
    )

    while True:
        user_input = input("\nAsk something: ")
        response = agent.invoke({"input": user_input})
        print("Final Answer:", response["output"])
        print("+++++++++++++++++")


if __name__ == "__main__":
    main()



Ask something:  What is 1 + 2 * 4




> Entering new AgentExecutor chain...
Thought: I need to calculate the expression 1 + 2 * 4. According to the order of operations, I should first perform the multiplication and then the addition. 

Action:
```
{
  "action": "Calculator",
  "action_input": "1 + 2 * 4"
}
```
...Running calculator tool

Observation: 9
Thought:I need to verify the calculation of the expression 1 + 2 * 4 to ensure accuracy. 

Action:
```
{
  "action": "Calculator",
  "action_input": "1 + 2 * 4"
}
```

...Running calculator tool

Observation: 9
Thought:I have confirmed the calculation of the expression 1 + 2 * 4. 

Thought: I now know the final answer.
Final Answer: 9

> Finished chain.
Final Answer: 9
+++++++++++++++++



Ask something:  What is the current weather condition in Moscow ?




> Entering new AgentExecutor chain...
Thought: I need to check the current weather conditions in Moscow.
Action:
```
{
  "action": "Weather",
  "action_input": "Moscow"
}
```
...Running weather tool

Observation: Temperature: 8.7°C, Wind Speed: 6.1 km/h
Thought:I now know the final answer
Final Answer: The current weather condition in Moscow is 8.7°C with a wind speed of 6.1 km/h.

> Finished chain.
Final Answer: The current weather condition in Moscow is 8.7°C with a wind speed of 6.1 km/h.
+++++++++++++++++



Ask something:  I am trying to calculate this math expression. What is one plus two times four ?




> Entering new AgentExecutor chain...
Thought: According to the order of operations, I need to perform the multiplication first and then the addition. So, I will calculate two times four first and then add one to the result. 
Action:
```
{
  "action": "Calculator",
  "action_input": "1 + 2 * 4"
}
```
...Running calculator tool

Observation: 9
Thought:I now know the final answer.  
Final Answer: 9

> Finished chain.
Final Answer: 9
+++++++++++++++++



Ask something:  I am living in Moscow. I was thinking of going to the park. I wonder how is weather outside. Is it going going to be nice today ?




> Entering new AgentExecutor chain...
Thought: I need to check the current weather in Moscow to determine if it's nice outside for a visit to the park. 
Action:
```
{
  "action": "Weather",
  "action_input": "Moscow"
}
```
...Running weather tool

Observation: Temperature: 8.7°C, Wind Speed: 6.1 km/h
Thought:The current temperature in Moscow is 8.7°C with a wind speed of 6.1 km/h. This suggests that the weather is relatively cool, but it may still be pleasant for a visit to the park, depending on personal preferences for temperature. 

Final Answer: The weather in Moscow is currently 8.7°C with a light wind, which may be nice for a park visit if you enjoy cooler temperatures.

> Finished chain.
Final Answer: The weather in Moscow is currently 8.7°C with a light wind, which may be nice for a park visit if you enjoy cooler temperatures.
+++++++++++++++++


KeyboardInterrupt: Interrupted by user

# Output from Trial Run

Ask something:  What is 1 + 2 * 4


> Entering new AgentExecutor chain...
Thought: I need to calculate the expression 1 + 2 * 4. According to the order of operations, I should first perform the multiplication and then the addition. 

Action:
```
{
  "action": "Calculator",
  "action_input": "1 + 2 * 4"
}
```
...Running calculator tool

Observation: 9
Thought:I need to verify the calculation of the expression 1 + 2 * 4 to ensure accuracy. 

Action:
```
{
  "action": "Calculator",
  "action_input": "1 + 2 * 4"
}
```

...Running calculator tool

Observation: 9
Thought:I have confirmed the calculation of the expression 1 + 2 * 4. 

Thought: I now know the final answer.
Final Answer: 9

> Finished chain.
Final Answer: 9


+++++++++++++++++

Ask something:  What is the current weather condition in Moscow ?


> Entering new AgentExecutor chain...
Thought: I need to check the current weather conditions in Moscow.
Action:
```
{
  "action": "Weather",
  "action_input": "Moscow"
}
```
...Running weather tool

Observation: Temperature: 8.7°C, Wind Speed: 6.1 km/h
Thought:I now know the final answer
Final Answer: The current weather condition in Moscow is 8.7°C with a wind speed of 6.1 km/h.

> Finished chain.
Final Answer: The current weather condition in Moscow is 8.7°C with a wind speed of 6.1 km/h.


+++++++++++++++++

Ask something:  I am trying to calculate this math expression. What is one plus two times four ?


> Entering new AgentExecutor chain...
Thought: According to the order of operations, I need to perform the multiplication first and then the addition. So, I will calculate two times four first and then add one to the result. 
Action:
```
{
  "action": "Calculator",
  "action_input": "1 + 2 * 4"
}
```
...Running calculator tool

Observation: 9
Thought:I now know the final answer.  
Final Answer: 9

> Finished chain.
Final Answer: 9


+++++++++++++++++

Ask something:  I am living in Moscow. I was thinking of going to the park. I wonder how is weather outside. Is it going going to be nice today ?


> Entering new AgentExecutor chain...
Thought: I need to check the current weather in Moscow to determine if it's nice outside for a visit to the park. 
Action:
```
{
  "action": "Weather",
  "action_input": "Moscow"
}
```
...Running weather tool

Observation: Temperature: 8.7°C, Wind Speed: 6.1 km/h
Thought:The current temperature in Moscow is 8.7°C with a wind speed of 6.1 km/h. This suggests that the weather is relatively cool, but it may still be pleasant for a visit to the park, depending on personal preferences for temperature. 

Final Answer: The weather in Moscow is currently 8.7°C with a light wind, which may be nice for a park visit if you enjoy cooler temperatures.

> Finished chain.
Final Answer: The weather in Moscow is currently 8.7°C with a light wind, which may be nice for a park visit if you enjoy cooler temperatures.


+++++++++++++++++